# Embedding + FAISS Checks

This notebook is for validating Phase 3 outputs before moving to Phase 4.

Goals:
- load saved claim embeddings
- load saved claim metadata
- load saved FAISS index
- inspect shapes and counts
- run nearest-neighbor checks
- inspect semantic quality manually
- later also support evidence snippet embeddings

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import faiss

In [2]:
# -----------------------------
# Claim artifacts
# -----------------------------
CLAIM_EMBEDDINGS_PATH = Path("../data/cache/embeddings/fever_claim_embeddings.npy")
CLAIM_METADATA_PATH = Path("../data/artifacts/faiss/fever_claim_metadata.pkl")
CLAIM_FAISS_INDEX_PATH = Path("../data/artifacts/faiss/fever_claims.index")

# -----------------------------
# Evidence artifacts
# -----------------------------
EVIDENCE_EMBEDDINGS_PATH = Path("../data/cache/embeddings/fever_evidence_embeddings.npy")
EVIDENCE_METADATA_PATH = Path("../data/artifacts/faiss/fever_evidence_metadata.pkl")
EVIDENCE_FAISS_INDEX_PATH = Path("../data/artifacts/faiss/fever_evidence.index")

In [3]:
# Load claim artifacts
claim_embeddings = np.load(CLAIM_EMBEDDINGS_PATH, allow_pickle=False)

with open(CLAIM_METADATA_PATH, "rb") as f:
    claim_metadata = pickle.load(f)

claim_index = faiss.read_index(str(CLAIM_FAISS_INDEX_PATH))

print("Claim embeddings shape:", claim_embeddings.shape)
print("Claim metadata length:", len(claim_metadata))
print("Claim FAISS ntotal:", claim_index.ntotal)

Claim embeddings shape: (165447, 384)
Claim metadata length: 165447
Claim FAISS ntotal: 165447


In [4]:
claim_embeddings.dtype

dtype('float32')

In [5]:
claim_metadata[0]

{'row_index': 0,
 'record_type': 'claim',
 'claim_id': 75397,
 'claim_text': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.',
 'claim_text_normalized': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.',
 'label': 'SUPPORTS',
 'verifiable': 'VERIFIABLE',
 'split': 'train'}

In [6]:
# Basic consistency checks
assert claim_embeddings.shape[0] == len(claim_metadata), "Mismatch between embeddings and metadata length"
assert claim_index.ntotal == len(claim_metadata), "Mismatch between FAISS ntotal and metadata length"

print("Claim artifact consistency checks passed.")

Claim artifact consistency checks passed.


In [7]:
def search_claim_neighbors(query_idx: int, top_k: int = 5):
    query_vec = np.ascontiguousarray(
        claim_embeddings[query_idx].reshape(1, -1).astype(np.float32)
    )
    distances, indices = claim_index.search(query_vec, top_k)

    query_meta = claim_metadata[query_idx]
    print("=" * 80)
    print("QUERY")
    print("row_index:", query_meta.get("row_index"))
    print("claim_id:", query_meta.get("claim_id"))
    print("label:", query_meta.get("label"))
    print("text:", query_meta.get("claim_text_normalized"))
    print("\nNEIGHBORS")

    for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        neighbor = claim_metadata[idx]
        print(f"{rank}. score={score:.4f}")
        print("   row_index:", neighbor.get("row_index"))
        print("   claim_id:", neighbor.get("claim_id"))
        print("   label:", neighbor.get("label"))
        print("   text:", neighbor.get("claim_text_normalized"))
        print()

In [8]:
search_claim_neighbors(0, top_k=5)

QUERY
row_index: 0
claim_id: 75397
label: SUPPORTS
text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.

NEIGHBORS
1. score=1.0000
   row_index: 0
   claim_id: 75397
   label: SUPPORTS
   text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.

2. score=0.9242
   row_index: 100377
   claim_id: 109854
   label: NOT ENOUGH INFO
   text: Nikolaj Coster-Waldau worked with the National Broadcasting Company.

3. score=0.9219
   row_index: 52409
   claim_id: 75398
   label: SUPPORTS
   text: Nikolaj Coster-Waldau was in multiple Fox Broadcasting Company productions.

4. score=0.8453
   row_index: 34018
   claim_id: 147655
   label: NOT ENOUGH INFO
   text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company multiple times including New Amsterdam and Virtuality.

5. score=0.8210
   row_index: 97331
   claim_id: 129329
   label: SUPPORTS
   text: Nikolaj Coster-Waldau worked



In [9]:
search_claim_neighbors(1, top_k=5)

QUERY
row_index: 1
claim_id: 150448
label: SUPPORTS
text: Roman Atwood is a content creator.

NEIGHBORS
1. score=1.0000
   row_index: 12935
   claim_id: 112756
   label: SUPPORTS
   text: Roman Atwood is a content creator.

2. score=1.0000
   row_index: 1
   claim_id: 150448
   label: SUPPORTS
   text: Roman Atwood is a content creator.

3. score=1.0000
   row_index: 54387
   claim_id: 143939
   label: SUPPORTS
   text: Roman Atwood is a content creator.

4. score=0.8276
   row_index: 36282
   claim_id: 161093
   label: SUPPORTS
   text: Roman Atwood is an artist.

5. score=0.7645
   row_index: 60821
   claim_id: 160382
   label: SUPPORTS
   text: Roman Atwood is a youtube vlogger.



In [10]:
search_claim_neighbors(2, top_k=5)

QUERY
row_index: 2
claim_id: 214861
label: SUPPORTS
text: History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.

NEIGHBORS
1. score=1.0000
   row_index: 26823
   claim_id: 214917
   label: NOT ENOUGH INFO
   text: History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.

2. score=1.0000
   row_index: 19105
   claim_id: 214862
   label: NOT ENOUGH INFO
   text: History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.

3. score=1.0000
   row_index: 17337
   claim_id: 214901
   label: NOT ENOUGH INFO
   text: History of art includes architecture, dance, sculpture, music, painting, poetry literature, theatre, narrative, film, photography and graphic arts.

4. score=1.0000
   row_index: 11543
   claim_id: 214890
   labe

In [11]:
# Inspect a few specific query positions
for idx in [0, 1, 2, 3, 4]:
    search_claim_neighbors(idx, top_k=5)

QUERY
row_index: 0
claim_id: 75397
label: SUPPORTS
text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.

NEIGHBORS
1. score=1.0000
   row_index: 0
   claim_id: 75397
   label: SUPPORTS
   text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.

2. score=0.9242
   row_index: 100377
   claim_id: 109854
   label: NOT ENOUGH INFO
   text: Nikolaj Coster-Waldau worked with the National Broadcasting Company.

3. score=0.9219
   row_index: 52409
   claim_id: 75398
   label: SUPPORTS
   text: Nikolaj Coster-Waldau was in multiple Fox Broadcasting Company productions.

4. score=0.8453
   row_index: 34018
   claim_id: 147655
   label: NOT ENOUGH INFO
   text: Nikolaj Coster-Waldau worked with the Fox Broadcasting Company multiple times including New Amsterdam and Virtuality.

5. score=0.8210
   row_index: 97331
   claim_id: 129329
   label: SUPPORTS
   text: Nikolaj Coster-Waldau worked

QUERY
row_index: 1
claim_id: 150448
label: SUPPORTS
text: Roman Atwood is a co

In [12]:
def get_claim_df_from_metadata(metadata_list):
    return pd.DataFrame(metadata_list)

claim_meta_df = get_claim_df_from_metadata(claim_metadata)
claim_meta_df.head()

,row_index,record_type,claim_id,claim_text,claim_text_normalized,label,verifiable,split
0,0,claim,75397,Nikolaj Coster-Waldau worked with the Fox Broa...,Nikolaj Coster-Waldau worked with the Fox Broa...,SUPPORTS,VERIFIABLE,train
1,1,claim,150448,Roman Atwood is a content creator.,Roman Atwood is a content creator.,SUPPORTS,VERIFIABLE,train
2,2,claim,214861,"History of art includes architecture, dance, s...","History of art includes architecture, dance, s...",SUPPORTS,VERIFIABLE,train
3,3,claim,156709,Adrienne Bailon is an accountant.,Adrienne Bailon is an accountant.,REFUTES,VERIFIABLE,train
4,4,claim,83235,System of a Down briefly disbanded in limbo.,System of a Down briefly disbanded in limbo.,NOT ENOUGH INFO,NOT VERIFIABLE,train


In [13]:
claim_meta_df["label"].value_counts(dropna=False)

label
SUPPORTS           86701
NOT ENOUGH INFO    42305
REFUTES            36441
Name: count, dtype: int64

In [14]:
claim_meta_df["split"].value_counts(dropna=False)

split
train    145449
dev        9999
test       9999
Name: count, dtype: int64

In [15]:
# Find duplicate exact normalized claim texts
duplicate_claims = claim_meta_df[
    claim_meta_df.duplicated(subset=["claim_text_normalized"], keep=False)
].sort_values("claim_text_normalized")

duplicate_claims.head(20)

,row_index,record_type,claim_id,claim_text,claim_text_normalized,label,verifiable,split
21210,21210,claim,159790,1.7% of water can be found in glaciers and ice...,1.7% of water can be found in glaciers and ice...,SUPPORTS,VERIFIABLE,train
101639,101639,claim,159760,1.7% of water can be found in glaciers and ice...,1.7% of water can be found in glaciers and ice...,SUPPORTS,VERIFIABLE,train
48189,48189,claim,159782,1.7% of water can be found in the ground.,1.7% of water can be found in the ground.,SUPPORTS,VERIFIABLE,train
131880,131880,claim,159765,1.7% of water can be found in the ground.,1.7% of water can be found in the ground.,SUPPORTS,VERIFIABLE,train
33155,33155,claim,212088,10 Cloverfield Lane stars an actor.,10 Cloverfield Lane stars an actor.,SUPPORTS,VERIFIABLE,train
111187,111187,claim,212082,10 Cloverfield Lane stars an actor.,10 Cloverfield Lane stars an actor.,SUPPORTS,VERIFIABLE,train
20004,20004,claim,6025,21 Jump Street grossed $201 billion.,21 Jump Street grossed $201 billion.,REFUTES,VERIFIABLE,train
124736,124736,claim,97387,21 Jump Street grossed $201 billion.,21 Jump Street grossed $201 billion.,REFUTES,VERIFIABLE,train
54894,54894,claim,90747,21 Jump Street is a television show.,21 Jump Street is a television show.,REFUTES,VERIFIABLE,train
86776,86776,claim,107804,21 Jump Street is a television show.,21 Jump Street is a television show.,REFUTES,VERIFIABLE,train


## Manual Quality Check Notes

Things to look for in nearest neighbors:
- exact duplicates should rank first
- close paraphrases should rank high
- same entity/topic variants should appear near each other
- clearly unrelated claims should not dominate top neighbors
- some false positives are normal with MiniLM

In [16]:
# Optional helper: search by claim text substring
def find_claim_rows(keyword: str, n: int = 10):
    mask = claim_meta_df["claim_text_normalized"].str.contains(keyword, case=False, na=False)
    return claim_meta_df[mask].head(n)

find_claim_rows("Roman Atwood")

,row_index,record_type,claim_id,claim_text,claim_text_normalized,label,verifiable,split
1,1,claim,150448,Roman Atwood is a content creator.,Roman Atwood is a content creator.,SUPPORTS,VERIFIABLE,train
5496,5496,claim,160397,Roman Atwood is a vlogger.,Roman Atwood is a vlogger.,SUPPORTS,VERIFIABLE,train
12443,12443,claim,89192,Roman Atwood is American.,Roman Atwood is American.,SUPPORTS,VERIFIABLE,train
12935,12935,claim,112756,Roman Atwood is a content creator.,Roman Atwood is a content creator.,SUPPORTS,VERIFIABLE,train
13230,13230,claim,118273,Roman Atwood is a car tire.,Roman Atwood is a car tire.,REFUTES,VERIFIABLE,train
14893,14893,claim,101480,Roman Atwood is a comedian.,Roman Atwood is a comedian.,SUPPORTS,VERIFIABLE,train
36282,36282,claim,161093,Roman Atwood is an artist.,Roman Atwood is an artist.,SUPPORTS,VERIFIABLE,train
40419,40419,claim,96013,Roman Atwood was born in 1983.,Roman Atwood was born in 1983.,SUPPORTS,VERIFIABLE,train
52389,52389,claim,119720,Roman Atwood was born in 1985.,Roman Atwood was born in 1985.,REFUTES,VERIFIABLE,train
54387,54387,claim,143939,Roman Atwood is a content creator.,Roman Atwood is a content creator.,SUPPORTS,VERIFIABLE,train


In [17]:
# Use one found row index for neighbor search
row_idx = claim_meta_df[claim_meta_df["claim_text_normalized"].str.contains("Roman Atwood", case=False, na=False)].index[0]
search_claim_neighbors(row_idx, top_k=5)

QUERY
row_index: 1
claim_id: 150448
label: SUPPORTS
text: Roman Atwood is a content creator.

NEIGHBORS
1. score=1.0000
   row_index: 12935
   claim_id: 112756
   label: SUPPORTS
   text: Roman Atwood is a content creator.

2. score=1.0000
   row_index: 1
   claim_id: 150448
   label: SUPPORTS
   text: Roman Atwood is a content creator.

3. score=1.0000
   row_index: 54387
   claim_id: 143939
   label: SUPPORTS
   text: Roman Atwood is a content creator.

4. score=0.8276
   row_index: 36282
   claim_id: 161093
   label: SUPPORTS
   text: Roman Atwood is an artist.

5. score=0.7645
   row_index: 60821
   claim_id: 160382
   label: SUPPORTS
   text: Roman Atwood is a youtube vlogger.



## Evidence Snippet Section

Run these cells only after the evidence snippet embeddings/index are built.

In [18]:
evidence_available = (
    EVIDENCE_EMBEDDINGS_PATH.exists()
    and EVIDENCE_METADATA_PATH.exists()
    and EVIDENCE_FAISS_INDEX_PATH.exists()
)

print("Evidence artifacts available:", evidence_available)

Evidence artifacts available: True


In [19]:
if evidence_available:
    evidence_embeddings = np.load(EVIDENCE_EMBEDDINGS_PATH, allow_pickle=False)

    with open(EVIDENCE_METADATA_PATH, "rb") as f:
        evidence_metadata = pickle.load(f)

    evidence_index = faiss.read_index(str(EVIDENCE_FAISS_INDEX_PATH))

    print("Evidence embeddings shape:", evidence_embeddings.shape)
    print("Evidence metadata length:", len(evidence_metadata))
    print("Evidence FAISS ntotal:", evidence_index.ntotal)
else:
    print("Evidence artifacts not built yet.")

Evidence embeddings shape: (40574, 384)
Evidence metadata length: 40574
Evidence FAISS ntotal: 40574


In [20]:
def search_evidence_neighbors(query_idx: int, top_k: int = 5):
    query_vec = np.ascontiguousarray(
        evidence_embeddings[query_idx].reshape(1, -1).astype(np.float32)
    )
    distances, indices = evidence_index.search(query_vec, top_k)

    query_meta = evidence_metadata[query_idx]
    print("=" * 80)
    print("QUERY EVIDENCE")
    print("claim_id:", query_meta.get("claim_id"))
    print("page_title:", query_meta.get("page_title_clean"))
    print("sentence_id:", query_meta.get("sentence_id"))
    print("text:", query_meta.get("snippet_text"))
    print("\nNEIGHBORS")

    for rank, (score, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        neighbor = evidence_metadata[idx]
        print(f"{rank}. score={score:.4f}")
        print("   claim_id:", neighbor.get("claim_id"))
        print("   page_title:", neighbor.get("page_title_clean"))
        print("   sentence_id:", neighbor.get("sentence_id"))
        print("   text:", neighbor.get("snippet_text"))
        print()

In [21]:
if evidence_available:
    search_evidence_neighbors(0, top_k=5)
else:
    print("Build evidence index first.")

QUERY EVIDENCE
claim_id: 75397
page_title: Nikolaj_Coster-Waldau
sentence_id: 7.0
text: He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot . Fox television Fox Broadcasting Company New Amsterdam New Amsterdam (TV series) Virtuality Virtuality (TV series)

NEIGHBORS
1. score=1.0000
   claim_id: 75397
   page_title: Nikolaj_Coster-Waldau
   sentence_id: 7.0
   text: He then played Detective John Amsterdam in the short-lived Fox television series New Amsterdam -LRB- 2008 -RRB- , as well as appearing as Frank Pike in the 2009 Fox television film Virtuality , originally intended as a pilot . Fox television Fox Broadcasting Company New Amsterdam New Amsterdam (TV series) Virtuality Virtuality (TV series)

2. score=0.5524
   claim_id: 154988
   page_title: New_Amsterdam_(TV_series)
   sentence_id: 1.0
   text: The series 

## Final checklist

Before moving to Phase 4, confirm:

- claim embeddings load successfully
- claim metadata aligns with embeddings
- FAISS claim search works
- nearest neighbors look semantically reasonable
- evidence snippet embeddings/index also work (after built)